# Trabajo Práctico - IA 2025
## Predicción de Costos Médicos Anuales
### Implementación con Clustering Sustractivo y Algoritmo Genético

**Estrategia:**
1. Clustering sustractivo para agrupar datos similares
2. Selección de datos representativos (30% train, 10% test por cluster)
3. Algoritmo genético para optimizar arquitectura de red feedforward
4. Entrenamiento final con 70% de datos completos

In [ ]:
# Importaciones necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.spatial import distance_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow version: {tf.__version__}")
print(f"Inicio de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Configurar semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

sns.set(style='whitegrid', palette='muted', font_scale=1.2)

## 1. Carga y Preparación de Datos

In [ ]:
# Cargar dataset
dirr = "/home/pedro_dev/Pedro/IA/competencia_FF/19-IA2025 medical_insurance.csv"
df = pd.read_csv(dirr)

print(f"Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
display(df.head())

# Verificar valores faltantes
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print(f"\nValores faltantes:")
    print(missing_values[missing_values > 0])

In [ ]:
# Preprocesamiento
# Convertir columnas categóricas
categorical_cols = ['sex', 'region', 'urban_rural', 'education', 'marital_status', 
                   'employment_status', 'smoker', 'alcohol_freq', 'plan_type', 'network_tier']

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Definir target y features
target = 'annual_medical_cost'
exclude_cols = ['person_id', target]
features = df.drop(columns=exclude_cols, errors='ignore')

# One-hot encoding
categorical_features = features.select_dtypes(include=['category', 'object']).columns.tolist()
features_encoded = pd.get_dummies(features, columns=categorical_features, drop_first=True)

# Rellenar NaN en features numéricas con la mediana
features_encoded = features_encoded.fillna(features_encoded.median())

X = features_encoded.values
y = df[target].values

print(f"\nFeatures finales: {X.shape[1]} columnas")
print(f"Rango del target: [{y.min():.2f}, {y.max():.2f}]")
print(f"Media del target: {y.mean():.2f}")

## 2. Modelo Baseline - Regresión Lineal
Este es el modelo que debemos superar

In [ ]:
# Entrenar modelo baseline de regresión lineal
X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model_baseline = LinearRegression()
model_baseline.fit(X_train_baseline, y_train_baseline)
y_pred_baseline = model_baseline.predict(X_test_baseline)

mae_baseline = mean_absolute_error(y_test_baseline, y_pred_baseline)
mse_baseline = mean_squared_error(y_test_baseline, y_pred_baseline)
r2_baseline = r2_score(y_test_baseline, y_pred_baseline)

print("=" * 60)
print("MODELO BASELINE - REGRESIÓN LINEAL")
print("=" * 60)
print(f"MAE (Mean Absolute Error): {mae_baseline:.4f}")
print(f"MSE (Mean Squared Error): {mse_baseline:.4f}")
print(f"RMSE: {np.sqrt(mse_baseline):.4f}")
print(f"R² Score: {r2_baseline:.4f}")
print("=" * 60)
print(f"\n*** OBJETIVO: Superar MAE < {mae_baseline:.4f} ***\n")

## 3. Clustering Sustractivo
Agrupar los datos en clusters para seleccionar los más representativos

In [ ]:
def subclust2(data, Ra, Rb=0, AcceptRatio=0.3, RejectRatio=0.1):
    """
    Clustering sustractivo.
    
    Parámetros:
    - data: matriz de datos (n_samples, n_features)
    - Ra: radio de influencia
    - Rb: radio de sustracción (por defecto Ra*1.15)
    - AcceptRatio: umbral de aceptación
    - RejectRatio: umbral de rechazo
    
    Retorna:
    - labels: etiquetas de cluster para cada punto
    - centers: centros de los clusters
    """
    if Rb == 0:
        Rb = Ra * 1.15
    
    # Normalizar datos
    scaler = MinMaxScaler()
    scaler.fit(data)
    ndata = scaler.transform(data)
    
    # Calcular potencial inicial
    P = distance_matrix(ndata, ndata)
    alpha = (Ra / 2) ** 2
    P = np.sum(np.exp(-P**2 / alpha), axis=0)
    
    centers = []
    i = np.argmax(P)
    C = ndata[i]
    p = P[i]
    centers = [C]
    
    continuar = True
    restarP = True
    
    while continuar:
        pAnt = p
        if restarP:
            P = P - p * np.array([np.exp(-np.linalg.norm(v - C)**2 / (Rb / 2)**2) for v in ndata])
        restarP = True
        
        i = np.argmax(P)
        C = ndata[i]
        p = P[i]
        
        if p > AcceptRatio * pAnt:
            centers = np.vstack((centers, C))
        elif p < RejectRatio * pAnt:
            continuar = False
        else:
            dr = np.min([np.linalg.norm(v - C) for v in centers])
            if dr / Ra + p / pAnt >= 1:
                centers = np.vstack((centers, C))
            else:
                P[i] = 0
                restarP = False
        
        if not any(v > 0 for v in P):
            continuar = False
    
    # Asignar cada punto al cluster más cercano
    distancias = [[np.linalg.norm(p - c) for p in ndata] for c in centers]
    labels = np.argmin(distancias, axis=0)
    
    # Desnormalizar centros
    centers = scaler.inverse_transform(centers)
    
    return labels, centers

In [ ]:
# Aplicar clustering sustractivo
print("Aplicando clustering sustractivo...")
start_time = time.time()

# Usar un subconjunto para el clustering si el dataset es muy grande
# Esto acelera el proceso sin perder representatividad
if len(X) > 50000:
    sample_indices = np.random.choice(len(X), 50000, replace=False)
    X_cluster = X[sample_indices]
    y_cluster = y[sample_indices]
else:
    X_cluster = X
    y_cluster = y

# Aplicar clustering con parámetros ajustados
labels, centers = subclust2(X_cluster, Ra=0.5, AcceptRatio=0.4, RejectRatio=0.15)

clustering_time = time.time() - start_time

n_clusters = len(centers)
print(f"\nClustering completado en {clustering_time:.2f} segundos")
print(f"Número de clusters encontrados: {n_clusters}")
print(f"\nDistribución de puntos por cluster:")
unique, counts = np.unique(labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} puntos ({100*count/len(labels):.2f}%)")

## 4. Selección de Datos Representativos
De cada cluster, seleccionar el 30% más representativo para entrenamiento y 10% para validación

In [ ]:
def select_representative_samples(X, y, labels, centers, train_ratio=0.30, test_ratio=0.10):
    """
    Selecciona los datos más representativos de cada cluster.
    
    Parámetros:
    - X: features
    - y: target
    - labels: etiquetas de cluster
    - centers: centros de clusters
    - train_ratio: porcentaje para entrenamiento
    - test_ratio: porcentaje para testing
    
    Retorna:
    - X_train_sel, X_test_sel, y_train_sel, y_test_sel
    """
    X_train_list = []
    X_test_list = []
    y_train_list = []
    y_test_list = []
    
    for cluster_id in range(len(centers)):
        # Obtener puntos del cluster
        cluster_mask = labels == cluster_id
        X_cluster = X[cluster_mask]
        y_cluster = y[cluster_mask]
        
        if len(X_cluster) == 0:
            continue
        
        # Calcular distancia al centro del cluster
        center = centers[cluster_id]
        distances = np.linalg.norm(X_cluster - center, axis=1)
        
        # Ordenar por cercanía al centro (más representativos)
        sorted_indices = np.argsort(distances)
        
        # Seleccionar los más cercanos
        n_train = max(1, int(len(X_cluster) * train_ratio))
        n_test = max(1, int(len(X_cluster) * test_ratio))
        
        train_indices = sorted_indices[:n_train]
        test_indices = sorted_indices[n_train:n_train + n_test]
        
        X_train_list.append(X_cluster[train_indices])
        X_test_list.append(X_cluster[test_indices])
        y_train_list.append(y_cluster[train_indices])
        y_test_list.append(y_cluster[test_indices])
    
    X_train_sel = np.vstack(X_train_list)
    X_test_sel = np.vstack(X_test_list)
    y_train_sel = np.concatenate(y_train_list)
    y_test_sel = np.concatenate(y_test_list)
    
    return X_train_sel, X_test_sel, y_train_sel, y_test_sel

# Seleccionar datos representativos
X_train_repr, X_test_repr, y_train_repr, y_test_repr = select_representative_samples(
    X_cluster, y_cluster, labels, centers, train_ratio=0.30, test_ratio=0.10
)

print(f"\nDatos seleccionados para algoritmo genético:")
print(f"  Train: {len(X_train_repr)} muestras ({100*len(X_train_repr)/len(X_cluster):.2f}% del subset)")
print(f"  Test: {len(X_test_repr)} muestras ({100*len(X_test_repr)/len(X_cluster):.2f}% del subset)")
print(f"  Total AG: {len(X_train_repr) + len(X_test_repr)} muestras")

## 5. Algoritmo Genético para Optimización de Hiperparámetros
Optimizar la arquitectura de la red feedforward

In [ ]:
class Individual:
    """
    Representa un individuo (arquitectura de red) en el algoritmo genético.
    
    Genes:
    - n_layers: número de capas ocultas (1-4)
    - nodes_per_layer: nodos por capa (lista)
    - use_batch_norm: si usar batch normalization (bool)
    - batch_norm_momentum: momentum de batch norm (0.8-0.99)
    - dropout_rates: tasa de dropout por capa (lista, 0.0-0.5)
    - activation: función de activación ('relu', 'tanh', 'elu')
    - learning_rate: tasa de aprendizaje (0.0001-0.01)
    """
    def __init__(self, n_features):
        self.n_features = n_features
        self.n_layers = np.random.randint(1, 5)  # 1 a 4 capas ocultas
        self.nodes_per_layer = [np.random.randint(16, 256) for _ in range(self.n_layers)]
        self.use_batch_norm = np.random.choice([True, False])
        self.batch_norm_momentum = np.random.uniform(0.8, 0.99) if self.use_batch_norm else 0.9
        self.dropout_rates = [np.random.uniform(0.0, 0.5) for _ in range(self.n_layers)]
        self.activation = np.random.choice(['relu', 'tanh', 'elu'])
        self.learning_rate = 10 ** np.random.uniform(-4, -2)  # 0.0001 a 0.01
        self.fitness = None
        self.training_time = None
    
    def mutate(self, mutation_rate=0.001):
        """Aplica mutación al individuo."""
        # Mutar número de capas
        if np.random.random() < mutation_rate:
            old_n_layers = self.n_layers
            self.n_layers = np.clip(self.n_layers + np.random.choice([-1, 1]), 1, 4)
            if self.n_layers > old_n_layers:
                self.nodes_per_layer.append(np.random.randint(16, 256))
                self.dropout_rates.append(np.random.uniform(0.0, 0.5))
            elif self.n_layers < old_n_layers:
                self.nodes_per_layer = self.nodes_per_layer[:-1]
                self.dropout_rates = self.dropout_rates[:-1]
        
        # Mutar nodos por capa
        for i in range(self.n_layers):
            if np.random.random() < mutation_rate:
                self.nodes_per_layer[i] = np.clip(
                    self.nodes_per_layer[i] + np.random.randint(-20, 21), 16, 256
                )
        
        # Mutar batch normalization
        if np.random.random() < mutation_rate:
            self.use_batch_norm = not self.use_batch_norm
        
        # Mutar momentum de batch norm
        if np.random.random() < mutation_rate and self.use_batch_norm:
            self.batch_norm_momentum = np.clip(
                self.batch_norm_momentum + np.random.uniform(-0.05, 0.05), 0.8, 0.99
            )
        
        # Mutar dropout rates
        for i in range(self.n_layers):
            if np.random.random() < mutation_rate:
                self.dropout_rates[i] = np.clip(
                    self.dropout_rates[i] + np.random.uniform(-0.1, 0.1), 0.0, 0.5
                )
        
        # Mutar activación
        if np.random.random() < mutation_rate:
            self.activation = np.random.choice(['relu', 'tanh', 'elu'])
        
        # Mutar learning rate
        if np.random.random() < mutation_rate:
            self.learning_rate = np.clip(
                self.learning_rate * np.random.uniform(0.5, 2.0), 0.0001, 0.01
            )
    
    def __repr__(self):
        return (f"Individual(layers={self.n_layers}, nodes={self.nodes_per_layer}, "
                f"bn={self.use_batch_norm}, dropout={[f'{d:.2f}' for d in self.dropout_rates]}, "
                f"act={self.activation}, lr={self.learning_rate:.5f}, fitness={self.fitness:.4f if self.fitness else 'N/A'})")

print("Clase Individual definida correctamente")

In [ ]:
def create_model(individual, input_dim):
    """
    Crea un modelo de red feedforward basado en los genes del individuo.
    """
    model = models.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    
    # Capas ocultas
    for i in range(individual.n_layers):
        model.add(layers.Dense(individual.nodes_per_layer[i], activation=individual.activation))
        
        if individual.use_batch_norm:
            model.add(layers.BatchNormalization(momentum=individual.batch_norm_momentum))
        
        if individual.dropout_rates[i] > 0.01:
            model.add(layers.Dropout(individual.dropout_rates[i]))
    
    # Capa de salida
    model.add(layers.Dense(1, activation='linear'))
    
    # Compilar
    optimizer = Adam(learning_rate=individual.learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    return model

print("Función create_model definida correctamente")

In [ ]:
def evaluate_individual(individual, X_train, y_train, X_test, y_test, 
                       mae_baseline, max_time_seconds=3600, epochs=50):
    """
    Evalúa el fitness de un individuo entrenando la red.
    
    Criterios:
    1. Debe entrenar en menos de max_time_seconds
    2. Debe tener MAE < mae_baseline
    3. Fitness = 1 / MAE (mayor es mejor)
    """
    try:
        start_time = time.time()
        
        # Normalizar datos
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Crear modelo
        model = create_model(individual, X_train.shape[1])
        
        # Early stopping para evitar sobreajuste
        early_stop = callbacks.EarlyStopping(
            monitor='val_loss', patience=10, restore_best_weights=True
        )
        
        # Entrenar con menos epochs para velocidad
        history = model.fit(
            X_train_scaled, y_train,
            validation_data=(X_test_scaled, y_test),
            epochs=epochs,
            batch_size=min(64, len(X_train) // 10),
            callbacks=[early_stop],
            verbose=0
        )
        
        # Predecir
        y_pred = model.predict(X_test_scaled, verbose=0).flatten()
        mae = mean_absolute_error(y_test, y_pred)
        
        training_time = time.time() - start_time
        
        # Verificar restricciones
        if training_time > max_time_seconds:
            fitness = 0.0  # Penalizar si toma mucho tiempo
        elif mae >= mae_baseline:
            fitness = 1.0 / mae_baseline  # Penalizar si no mejora el baseline
        else:
            fitness = 1.0 / mae  # Fitness normal
        
        individual.fitness = fitness
        individual.training_time = training_time
        individual.mae = mae
        
        # Limpiar memoria
        del model
        tf.keras.backend.clear_session()
        
        return fitness
        
    except Exception as e:
        print(f"Error evaluando individuo: {e}")
        individual.fitness = 0.0
        individual.training_time = 0
        individual.mae = float('inf')
        return 0.0

print("Función evaluate_individual definida correctamente")

In [ ]:
def crossover(parent1, parent2):
    """
    Crea un nuevo individuo combinando genes de dos padres.
    """
    child = Individual(parent1.n_features)
    
    # Heredar estructura de capas del mejor padre
    if parent1.fitness > parent2.fitness:
        child.n_layers = parent1.n_layers
        child.nodes_per_layer = parent1.nodes_per_layer.copy()
        child.dropout_rates = parent1.dropout_rates.copy()
    else:
        child.n_layers = parent2.n_layers
        child.nodes_per_layer = parent2.nodes_per_layer.copy()
        child.dropout_rates = parent2.dropout_rates.copy()
    
    # Mezclar otros hiperparámetros
    child.use_batch_norm = np.random.choice([parent1.use_batch_norm, parent2.use_batch_norm])
    child.batch_norm_momentum = np.random.choice([parent1.batch_norm_momentum, parent2.batch_norm_momentum])
    child.activation = np.random.choice([parent1.activation, parent2.activation])
    child.learning_rate = np.random.choice([parent1.learning_rate, parent2.learning_rate])
    
    return child

print("Función crossover definida correctamente")

In [ ]:
def genetic_algorithm(X_train, y_train, X_test, y_test, mae_baseline,
                     population_size=100, elite_percentage=0.10, 
                     mutation_rate=0.001, generations=20, max_time_per_model=3600):
    """
    Algoritmo genético para optimizar la arquitectura de la red.
    
    Parámetros:
    - population_size: tamaño de la población
    - elite_percentage: porcentaje de elite que pasa directamente
    - mutation_rate: tasa de mutación
    - generations: número de generaciones
    """
    n_features = X_train.shape[1]
    elite_size = int(population_size * elite_percentage)
    
    # Inicializar población
    print(f"\nInicializando población de {population_size} individuos...")
    population = [Individual(n_features) for _ in range(population_size)]
    
    # Histórico para visualización
    history = {
        'best_fitness': [],
        'avg_fitness': [],
        'best_mae': [],
        'best_individual': []
    }
    
    print(f"\n{'='*80}")
    print(f"INICIANDO ALGORITMO GENÉTICO")
    print(f"{'='*80}")
    print(f"Población: {population_size} | Elite: {elite_size} ({elite_percentage*100:.0f}%)")
    print(f"Mutación: {mutation_rate} | Generaciones: {generations}")
    print(f"MAE Baseline a superar: {mae_baseline:.4f}")
    print(f"{'='*80}\n")
    
    ga_start_time = time.time()
    
    for gen in range(generations):
        gen_start_time = time.time()
        print(f"\n--- Generación {gen + 1}/{generations} ---")
        
        # Evaluar población
        for idx, individual in enumerate(population):
            if individual.fitness is None:
                evaluate_individual(individual, X_train, y_train, X_test, y_test, 
                                  mae_baseline, max_time_seconds=max_time_per_model, 
                                  epochs=30)  # Menos epochs para velocidad
                
                if (idx + 1) % 10 == 0:
                    print(f"  Evaluados: {idx + 1}/{population_size}", end='\r')
        
        # Ordenar por fitness
        population.sort(key=lambda x: x.fitness, reverse=True)
        
        # Estadísticas
        best_individual = population[0]
        best_fitness = best_individual.fitness
        avg_fitness = np.mean([ind.fitness for ind in population])
        best_mae = best_individual.mae
        
        history['best_fitness'].append(best_fitness)
        history['avg_fitness'].append(avg_fitness)
        history['best_mae'].append(best_mae)
        history['best_individual'].append(best_individual)
        
        gen_time = time.time() - gen_start_time
        print(f"\n  Mejor Fitness: {best_fitness:.6f} | MAE: {best_mae:.4f} | "
              f"Tiempo: {gen_time:.1f}s")
        print(f"  Mejor individuo: {best_individual.n_layers} capas, "
              f"nodos={best_individual.nodes_per_layer}, act={best_individual.activation}")
        
        # Selección y reproducción
        new_population = []
        
        # Elite pasa directamente
        new_population.extend(population[:elite_size])
        
        # Generar resto mediante crossover y mutación
        while len(new_population) < population_size:
            # Selección por torneo
            parent1 = max(np.random.choice(population[:population_size//2], 3), 
                         key=lambda x: x.fitness)
            parent2 = max(np.random.choice(population[:population_size//2], 3), 
                         key=lambda x: x.fitness)
            
            # Crossover
            child = crossover(parent1, parent2)
            
            # Mutación
            child.mutate(mutation_rate)
            
            new_population.append(child)
        
        population = new_population
    
    total_ga_time = time.time() - ga_start_time
    
    print(f"\n{'='*80}")
    print(f"ALGORITMO GENÉTICO COMPLETADO")
    print(f"{'='*80}")
    print(f"Tiempo total: {total_ga_time/60:.2f} minutos")
    print(f"Mejor MAE alcanzado: {history['best_mae'][-1]:.4f}")
    print(f"Mejora sobre baseline: {((mae_baseline - history['best_mae'][-1])/mae_baseline*100):.2f}%")
    print(f"{'='*80}\n")
    
    return history['best_individual'][-1], history

print("Función genetic_algorithm definida correctamente")

## 6. Ejecutar Algoritmo Genético

In [ ]:
# Ejecutar algoritmo genético
print("\n" + "="*80)
print("FASE 1: OPTIMIZACIÓN CON ALGORITMO GENÉTICO")
print("="*80)

best_individual, ga_history = genetic_algorithm(
    X_train_repr, y_train_repr, X_test_repr, y_test_repr,
    mae_baseline=mae_baseline,
    population_size=100,
    elite_percentage=0.10,
    mutation_rate=0.001,
    generations=15,  # Ajustar según tiempo disponible
    max_time_per_model=3600
)

print("\n" + "="*80)
print("MEJOR INDIVIDUO ENCONTRADO")
print("="*80)
print(f"Capas ocultas: {best_individual.n_layers}")
print(f"Nodos por capa: {best_individual.nodes_per_layer}")
print(f"Batch Normalization: {best_individual.use_batch_norm}")
if best_individual.use_batch_norm:
    print(f"  - Momentum: {best_individual.batch_norm_momentum:.4f}")
print(f"Dropout rates: {[f'{d:.3f}' for d in best_individual.dropout_rates]}")
print(f"Activación: {best_individual.activation}")
print(f"Learning rate: {best_individual.learning_rate:.6f}")
print(f"MAE en validación: {best_individual.mae:.4f}")
print(f"Fitness: {best_individual.fitness:.6f}")
print("="*80)

## 7. Visualización de la Evolución del Algoritmo Genético

In [ ]:
# Graficar evolución
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Fitness
axes[0].plot(ga_history['best_fitness'], 'b-', linewidth=2, label='Mejor Fitness')
axes[0].plot(ga_history['avg_fitness'], 'r--', linewidth=2, label='Fitness Promedio')
axes[0].set_xlabel('Generación')
axes[0].set_ylabel('Fitness')
axes[0].set_title('Evolución del Fitness por Generación')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(ga_history['best_mae'], 'g-', linewidth=2, label='Mejor MAE')
axes[1].axhline(y=mae_baseline, color='r', linestyle='--', linewidth=2, label='MAE Baseline')
axes[1].set_xlabel('Generación')
axes[1].set_ylabel('MAE')
axes[1].set_title('Evolución del MAE por Generación')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('genetic_algorithm_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGráfico guardado como 'genetic_algorithm_evolution.png'")

## 8. Entrenamiento Final con Todo el Dataset
Entrenar el mejor modelo con 70% de los datos completos y evaluar con 30%

In [ ]:
print("\n" + "="*80)
print("FASE 2: ENTRENAMIENTO FINAL CON DATASET COMPLETO")
print("="*80)

# Dividir dataset completo
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f"\nDatos de entrenamiento final: {len(X_train_final)} muestras (70%)")
print(f"Datos de test final: {len(X_test_final)} muestras (30%)")

# Normalizar
scaler_final = StandardScaler()
X_train_final_scaled = scaler_final.fit_transform(X_train_final)
X_test_final_scaled = scaler_final.transform(X_test_final)

# Crear modelo final
print("\nCreando modelo final con arquitectura óptima...")
final_model = create_model(best_individual, X_train_final.shape[1])

print("\nResumen del modelo:")
final_model.summary()

# Callbacks
early_stop_final = callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
)

# Entrenar
print("\nEntrenando modelo final...")
start_train_time = time.time()

history_final = final_model.fit(
    X_train_final_scaled, y_train_final,
    validation_data=(X_test_final_scaled, y_test_final),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop_final, reduce_lr],
    verbose=1
)

train_time_final = time.time() - start_train_time
print(f"\nTiempo de entrenamiento final: {train_time_final/60:.2f} minutos")

In [ ]:
# Evaluación final
print("\n" + "="*80)
print("EVALUACIÓN FINAL")
print("="*80)

y_pred_final = final_model.predict(X_test_final_scaled, verbose=0).flatten()

mae_final = mean_absolute_error(y_test_final, y_pred_final)
mse_final = mean_squared_error(y_test_final, y_pred_final)
rmse_final = np.sqrt(mse_final)
r2_final = r2_score(y_test_final, y_pred_final)

print("\nRESULTADOS MODELO FINAL:")
print(f"  MAE:  {mae_final:.4f}")
print(f"  MSE:  {mse_final:.4f}")
print(f"  RMSE: {rmse_final:.4f}")
print(f"  R²:   {r2_final:.4f}")

print("\nCOMPARACIÓN CON BASELINE:")
print(f"  MAE Baseline:  {mae_baseline:.4f}")
print(f"  MAE Final:     {mae_final:.4f}")
print(f"  Mejora:        {((mae_baseline - mae_final)/mae_baseline*100):.2f}%")

if mae_final < mae_baseline:
    print("\n✓ OBJETIVO CUMPLIDO: El modelo supera al baseline")
else:
    print("\n✗ ATENCIÓN: El modelo no supera al baseline")

print("="*80)

In [ ]:
# Visualización de resultados finales
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Pérdida durante entrenamiento
axes[0, 0].plot(history_final.history['loss'], label='Train Loss')
axes[0, 0].plot(history_final.history['val_loss'], label='Validation Loss')
axes[0, 0].set_xlabel('Época')
axes[0, 0].set_ylabel('Loss (MSE)')
axes[0, 0].set_title('Evolución de la Pérdida')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MAE durante entrenamiento
axes[0, 1].plot(history_final.history['mae'], label='Train MAE')
axes[0, 1].plot(history_final.history['val_mae'], label='Validation MAE')
axes[0, 1].axhline(y=mae_baseline, color='r', linestyle='--', label='Baseline MAE')
axes[0, 1].set_xlabel('Época')
axes[0, 1].set_ylabel('MAE')
axes[0, 1].set_title('Evolución del MAE')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Predicciones vs Real
axes[1, 0].scatter(y_test_final, y_pred_final, alpha=0.5, s=10)
axes[1, 0].plot([y_test_final.min(), y_test_final.max()], 
                [y_test_final.min(), y_test_final.max()], 
                'r--', lw=2)
axes[1, 0].set_xlabel('Costo Real')
axes[1, 0].set_ylabel('Costo Predicho')
axes[1, 0].set_title('Predicciones vs Valores Reales')
axes[1, 0].grid(True, alpha=0.3)

# Distribución de errores
errors = y_test_final - y_pred_final
axes[1, 1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1, 1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Error (Real - Predicho)')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].set_title('Distribución de Errores')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('final_model_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGráfico guardado como 'final_model_results.png'")

## 9. Resumen de Tiempos de Ejecución

In [ ]:
# Calcular tiempo total
print("\n" + "="*80)
print("RESUMEN DE TIEMPOS DE EJECUCIÓN")
print("="*80)
print(f"Fin de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nNota: Para obtener el tiempo total exacto, ejecutar todas las celdas desde el inicio.")
print("="*80)

## 10. Guardar Modelo Final

In [ ]:
# Guardar modelo y scaler
final_model.save('modelo_final_ff.h5')
print("Modelo guardado como 'modelo_final_ff.h5'")

# Guardar información del mejor individuo
import json

best_config = {
    'n_layers': best_individual.n_layers,
    'nodes_per_layer': best_individual.nodes_per_layer,
    'use_batch_norm': best_individual.use_batch_norm,
    'batch_norm_momentum': best_individual.batch_norm_momentum,
    'dropout_rates': best_individual.dropout_rates,
    'activation': best_individual.activation,
    'learning_rate': best_individual.learning_rate,
    'mae_validation': float(best_individual.mae),
    'mae_final': float(mae_final),
    'mae_baseline': float(mae_baseline),
    'improvement': float((mae_baseline - mae_final) / mae_baseline * 100)
}

with open('best_model_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

print("Configuración guardada como 'best_model_config.json'")
print("\n¡Implementación completada exitosamente!")